# Test MCP Servers via MaaS Gateway

This notebook tests MCP server access through the MaaS gateway:
1. MCP endpoint discovery via gateway
2. Authentication enforcement
3. Streamable HTTP connectivity per server
4. Tool invocation test
5. Context7 (external) integration

**Prerequisites:**
- MaaS enabled with MCP servers registered (`2_enable_maas.ipynb` completed)
- At least one MCP server deployed in `mcp-servers` namespace

In [ ]:
import subprocess
import json
import os

result = subprocess.run(
    ["oc", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()

# MaaS gateway hostname from the Gateway listener config
MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"

# Use API key if available, otherwise fall back to OCP token
API_KEY = os.getenv("MAAS_API_KEY", "")
if not API_KEY:
    token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    API_KEY = token_result.stdout.strip()
    print("Using OpenShift token for authentication")
else:
    print(f"Using MaaS API key: {API_KEY[:15]}...")

print(f"\n\u2705 MaaS Gateway: {MAAS_HOST}")

## 1. Discover MCP Endpoints

List all MCP servers registered with the MaaS gateway via HTTPRoute.

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.${CLUSTER_DOMAIN}"

echo "MCP Servers via MaaS Gateway"
echo "============================================================"
echo ""

printf "%-25s %-55s %s\n" "SERVER" "GATEWAY ENDPOINT" "STATUS"
printf "%-25s %-55s %s\n" "-------" "----------------" "------"

for route in $(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    SHORT_NAME=$(echo $route | sed 's/^mcp-route-//')
    URL="${HOST}/mcp/${SHORT_NAME}/mcp"
    printf "%-25s %-55s" "${SHORT_NAME}" "${URL}"
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
      -H "Authorization: Bearer $(oc whoami -t)" "${URL}")
    if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
        printf " ✅\n"
    else
        printf " ⚠️ (HTTP %s)\n" "$HTTP_CODE"
    fi
done

if [ -z "$(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{.items[*].metadata.name}' 2>/dev/null)" ]; then
    echo "⚠️  No MCP HTTPRoutes registered. Run 2_enable_maas.ipynb Step 8 first."
fi

## 2. Test Authentication Enforcement

Verify that MCP endpoints via the gateway require a valid API key.

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.${CLUSTER_DOMAIN}"

# Pick first MCP route
FIRST_ROUTE=$(oc get httproute -n ${MCP_NS} -l maas.opendatahub.io/managed=true -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)
SHORT_NAME=$(echo $FIRST_ROUTE | sed 's/^mcp-route-//')
URL="${HOST}/mcp/${SHORT_NAME}/mcp"

echo "Testing auth enforcement on: ${URL}"
echo ""

# Test without auth
echo "1. No auth header (expecting 401/403):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${URL}")
if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "   \u2705 Rejected (HTTP ${HTTP_CODE}) — auth enforced"
else
    echo "   \u26a0\ufe0f  Got HTTP ${HTTP_CODE} — auth may not be enforced"
fi

echo ""

# Test with invalid key
echo "2. Invalid API key (expecting 401/403):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
  -H "Authorization: Bearer sk-oai-INVALID-KEY" "${URL}")
if [ "$HTTP_CODE" = "401" ] || [ "$HTTP_CODE" = "403" ]; then
    echo "   \u2705 Rejected (HTTP ${HTTP_CODE}) — invalid key rejected"
else
    echo "   \u26a0\ufe0f  Got HTTP ${HTTP_CODE}"
fi

echo ""

# Test with valid token
echo "3. Valid OCP token (expecting 200/405):"
HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 \
  -H "Authorization: Bearer $(oc whoami -t)" "${URL}")
if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "   \u2705 Accepted (HTTP ${HTTP_CODE}) — valid auth works"
else
    echo "   \u26a0\ufe0f  Got HTTP ${HTTP_CODE}"
fi

## 3. Test MCP Protocol — Sequential Thinking

Send an MCP `initialize` request to the Sequential Thinking server via the gateway using Streamable HTTP.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/sequential-thinking/mcp"

echo "Testing MCP protocol on Sequential Thinking server..."
echo "Endpoint: ${URL}"
echo ""

# Streamable HTTP POST test — send initialize request
echo "Streamable HTTP POST test:"
RESPONSE=$(curl -sSk -m 5 -X POST \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}' \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "protocolVersion\|serverInfo\|jsonrpc"; then
    echo "✅ Streamable HTTP active — server responded to initialize"
    echo "   Response (first 300 chars):"
    echo "${RESPONSE:0:300}"
else
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 -X POST \
      -H "Authorization: Bearer $(oc whoami -t)" \
      -H "Content-Type: application/json" \
      -d '{}' "${URL}")
    echo "⚠️ HTTP ${HTTP_CODE} — Response: ${RESPONSE:0:200}"
fi

## 4. Test MCP Protocol — GitHub

Verify the GitHub MCP server is accessible through the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/github/mcp"

echo "Testing MCP protocol on GitHub server..."
echo "Endpoint: ${URL}"
echo ""

# Streamable HTTP POST test — send initialize request
echo "Streamable HTTP POST test:"
RESPONSE=$(curl -sSk -m 5 -X POST \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}' \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "protocolVersion\|serverInfo\|jsonrpc"; then
    echo "✅ Streamable HTTP active — server responded to initialize"
    echo "   Response (first 300 chars):"
    echo "${RESPONSE:0:300}"
else
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 -X POST \
      -H "Authorization: Bearer $(oc whoami -t)" \
      -H "Content-Type: application/json" \
      -d '{}' "${URL}")
    echo "⚠️ HTTP ${HTTP_CODE} — Response: ${RESPONSE:0:200}"
fi

## 5. Test MCP Protocol — gh-grep

Verify the gh-grep MCP server is accessible through the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/gh-grep/mcp"

echo "Testing MCP protocol on gh-grep server..."
echo "Endpoint: ${URL}"
echo ""

# Streamable HTTP POST test — send initialize request
echo "Streamable HTTP POST test:"
RESPONSE=$(curl -sSk -m 5 -X POST \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}' \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "protocolVersion\|serverInfo\|jsonrpc"; then
    echo "✅ Streamable HTTP active — server responded to initialize"
    echo "   Response (first 300 chars):"
    echo "${RESPONSE:0:300}"
else
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 -X POST \
      -H "Authorization: Bearer $(oc whoami -t)" \
      -H "Content-Type: application/json" \
      -d '{}' "${URL}")
    echo "⚠️ HTTP ${HTTP_CODE} — Response: ${RESPONSE:0:200}"
fi

## 6. Test MCP Protocol — Code Sandbox

Verify the Code Sandbox MCP server (secure Python/Bash/Node execution) is accessible through the gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.${CLUSTER_DOMAIN}"
URL="${HOST}/mcp/code-sandbox/mcp"

echo "Testing MCP protocol on Code Sandbox server..."
echo "Endpoint: ${URL}"
echo ""

# Health check (Code Sandbox exposes /health)
echo "Health check:"
HEALTH=$(curl -sSk -m 5 \
  -H "Authorization: Bearer $(oc whoami -t)" \
  "${HOST}/mcp/code-sandbox/health" 2>&1)
echo "  $HEALTH"
echo ""

# Streamable HTTP POST test — send initialize request
echo "Streamable HTTP POST test:"
RESPONSE=$(curl -sSk -m 5 -X POST \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2025-03-26","capabilities":{},"clientInfo":{"name":"test","version":"1.0"}}}' \
  "${URL}" 2>&1 || true)

if echo "$RESPONSE" | grep -q "protocolVersion\|serverInfo\|jsonrpc"; then
    echo "✅ Streamable HTTP active — server responded to initialize"
    echo "   Response (first 300 chars):"
    echo "${RESPONSE:0:300}"
else
    HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 -X POST \
      -H "Authorization: Bearer $(oc whoami -t)" \
      -H "Content-Type: application/json" \
      -d '{}' "${URL}")
    echo "⚠️ HTTP ${HTTP_CODE} — Response: ${RESPONSE:0:200}"
fi

## 7. Comparison: Direct Route vs MaaS Gateway

Compare accessing MCP servers directly via OpenShift Routes vs through the MaaS gateway.

In [ ]:
%%bash
MCP_NS="mcp-servers"
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
MAAS_HOST="https://maas-api.${CLUSTER_DOMAIN}"

echo "Direct Route vs MaaS Gateway"
echo "============================================================"
echo ""
printf "%-30s %-12s %-12s\n" "SERVER" "DIRECT" "VIA MAAS"
printf "%-30s %-12s %-12s\n" "-------" "------" "--------"

for route in $(oc get routes -n ${MCP_NS} -o jsonpath='{range .items[*]}{.metadata.name}{"\n"}{end}' 2>/dev/null); do
    host=$(oc get route $route -n ${MCP_NS} -o jsonpath='{.spec.host}')
    DIRECT_URL="https://${host}/mcp"
    SHORT_NAME=$(echo $route | sed 's/^mcp-//')
    MAAS_URL="${MAAS_HOST}/mcp/${SHORT_NAME}/mcp"

    # Direct access (no auth)
    DIRECT_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${DIRECT_URL}")

    # MaaS access (no auth — should be blocked)
    MAAS_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" -m 5 "${MAAS_URL}")

    printf "%-30s HTTP %-6s  HTTP %-6s\n" "${route}" "${DIRECT_CODE}" "${MAAS_CODE}"
done

echo ""
echo "Expected: Direct routes return 200/405 (open), MaaS routes return 401/403 (auth required)"
echo ""
echo "✅ MaaS gateway enforces authentication on MCP tool access"

## Summary

| Test | What It Validates |
|------|-------------------|
| Endpoint Discovery | MCP servers registered via HTTPRoute and accessible |
| Auth Enforcement | Gateway rejects unauthenticated/invalid MCP requests |
| Streamable HTTP | MCP protocol (Streamable HTTP) works through gateway |
| Direct vs Gateway | MaaS adds auth layer vs open direct Routes |

### MCP Server Endpoints (via MaaS Gateway)

| Server | Gateway Path | Offline |
|--------|--------------|---------|
| Sequential Thinking | `https://maas-api.<domain>/mcp/sequential-thinking/mcp` | ✅ |
| Code Sandbox | `https://maas-api.<domain>/mcp/code-sandbox/mcp` | ✅ |
| Context7 | `https://maas-api.<domain>/mcp/context7/mcp` | ❌ |
| GitHub | `https://maas-api.<domain>/mcp/github/mcp` | ❌ |
| gh-grep | `https://maas-api.<domain>/mcp/gh-grep/mcp` | ❌ |
| Playwright | `https://maas-api.<domain>/mcp/playwright/mcp` | ❌ |

## Next Steps

→ `../3_run_and_control/1_ide_configuration.ipynb` — Configure your IDE to use MaaS for both models and MCP tools
→ `../3_run_and_control/3_maas_advanced.ipynb` — Explore subscriptions, rate limit tuning, and monitoring